In [6]:
from pathlib import Path
import sys

from flowrec.item_representation.content_embeddings import ItemEmbeddingStore
from flowrec.item_representation.semantic_id.rqvae import (
    RQVAEConfig,
    RQVAETrainConfig)
from flowrec.item_representation.semantic_id.rqvae_tokenizer import  RQVAEItemTokenizer, RQVAEItemTokenizerConfig

   

In [7]:

embeddings_path = Path("../artifacts/item_embeddings/beauty_content_embeddings.pt")
output_dir = Path("../artifacts/tokenizers/rqvae_beauty")
output_dir.mkdir(parents=True, exist_ok=True)

In [8]:
embedding_store = ItemEmbeddingStore.load(embeddings_path)

item_ids = embedding_store.item_ids
embeddings = embedding_store.embeddings

print("Number of items:", len(item_ids))
print("Embedding shape:", embeddings.shape)

Number of items: 259204
Embedding shape: torch.Size([259204, 384])


In [9]:
rqvae_cfg = RQVAEConfig(
    input_dim=embeddings.shape[1],
    latent_dim=32,
    hidden_dims=(512, 256, 128),
    num_codebooks=3,
    codebook_size=256,
    commitment_weight=0.25,
    dropout=0.0,
    use_batch_norm=False,
)

train_cfg = RQVAETrainConfig(
    epochs=50,
    batch_size=1024,
    learning_rate=1e-3,
    weight_decay=0.0,
    val_ratio=0.05,
    seed=42,
    gradient_clip_norm=1.0,
    device="cpu",  # use "cpu" if needed
)

tokenizer_cfg = RQVAEItemTokenizerConfig(
    rqvae=rqvae_cfg,
    train=train_cfg,

    # Recommended for text embeddings.
    normalize="l2",

    # Important for TIGER-style flat token vocabulary:
    # code_l -> code_l + l * codebook_size
    level_offset_tokens=True,

    # Resolve duplicate semantic IDs by appending a suffix token.
    collision_policy="append",

    encode_batch_size=4096,
)


In [10]:
tokenizer = RQVAEItemTokenizer(tokenizer_cfg)

tokenizer.fit(
    embeddings=embeddings,
    item_ids=item_ids,
)

In [11]:
tokenizer.save(output_dir)

semantic_ids_path = output_dir / "semantic_ids.json"

print("Saved tokenizer to:", output_dir)
print("Saved semantic IDs to:", semantic_ids_path)


Saved tokenizer to: ../artifacts/tokenizers/rqvae_beauty
Saved semantic IDs to: ../artifacts/tokenizers/rqvae_beauty/semantic_ids.json


In [12]:
semantic_id_store = tokenizer.semantic_id_store
semantic_id_mapping = semantic_id_store.to_plain_dict()

print("Number of semantic IDs:", semantic_id_store.num_items)
print("Semantic ID length:", semantic_id_store.sid_length)
print("Unique semantic IDs:", semantic_id_store.unique_code_count)
print("Collisions after resolving:", semantic_id_store.collision_count)

for item_id in item_ids[:5]:
    print(item_id, "->", semantic_id_mapping[item_id])

Number of semantic IDs: 259204
Semantic ID length: 4
Unique semantic IDs: 259204
Collisions after resolving: 0
0205616461 -> [140, 462, 638, 768]
0558925278 -> [140, 462, 638, 769]
0733001998 -> [140, 462, 638, 770]
0737104473 -> [140, 462, 638, 771]
0762451459 -> [140, 462, 638, 772]
